# DSPy ChainOfThought Deep Dive

The `ChainOfThought` module adds step-by-step reasoning to language model calls.

**What You'll Learn:**
- How ChainOfThought adds reasoning to predictions
- Inspecting the rationale field
- When CoT helps vs hurts performance
- Combining CoT with custom modules

## Setup

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, print_step, print_result, print_error, configure_dspy
from dotenv import load_dotenv

load_dotenv('../../.env')

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=1000)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")
    print("Make sure you have set your OPENAI_API_KEY in the .env file")

## Example 1: Basic ChainOfThought

ChainOfThought automatically adds a `reasoning` field to the output.

In [ ]:
class MathSolver(dspy.Signature):
    """Solve the math problem."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

cot = dspy.ChainOfThought(MathSolver)
result = cot(problem="A store has 45 apples. If 3/5 are sold, how many remain?")

print_result(
    f"Problem: A store has 45 apples. If 3/5 are sold, how many remain?\n\n"
    f"Reasoning: {result.reasoning}\n\n"
    f"Answer: {result.answer}",
    "Math with CoT"
)

## Example 2: The Reasoning Field

The reasoning traces how the model arrived at its answer.

In [ ]:
class FactChecker(dspy.Signature):
    """Determine if the statement is true or false."""
    statement = dspy.InputField(desc="A factual claim to verify")
    verdict = dspy.OutputField(desc="Either 'true' or 'false'")

checker = dspy.ChainOfThought(FactChecker)

statements = [
    "The Great Wall of China is visible from space with the naked eye.",
    "Water boils at 100 degrees Celsius at sea level.",
    "Lightning never strikes the same place twice."
]

for stmt in statements:
    result = checker(statement=stmt)
    print(f"\nStatement: {stmt}")
    print(f"Reasoning: {result.reasoning[:150]}...")
    print(f"Verdict: {result.verdict}")

## Example 3: When CoT Helps

Comparing Predict vs ChainOfThought on a tricky logic problem.

In [ ]:
class LogicProblem(dspy.Signature):
    """Solve this logic problem carefully."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

tricky_problem = "If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?"

# Without reasoning
simple = dspy.Predict(LogicProblem)
simple_result = simple(problem=tricky_problem)
print(f"Predict answer: {simple_result.answer}")

# With reasoning
reasoned = dspy.ChainOfThought(LogicProblem)
reasoned_result = reasoned(problem=tricky_problem)
print(f"CoT reasoning: {reasoned_result.reasoning[:200]}...")
print(f"CoT answer: {reasoned_result.answer}")

## Example 4: When CoT May Not Help

For simple tasks, CoT adds unnecessary tokens without improving accuracy.

In [ ]:
class SimpleClassifier(dspy.Signature):
    """Classify the text as positive or negative."""
    text = dspy.InputField()
    sentiment = dspy.OutputField(desc="positive or negative")

simple_cls = dspy.Predict(SimpleClassifier)
cot_cls = dspy.ChainOfThought(SimpleClassifier)

test_text = "I love this product! It's amazing!"

simple_r = simple_cls(text=test_text)
cot_r = cot_cls(text=test_text)

print(f"Text: {test_text}")
print(f"Predict: {simple_r.sentiment} (direct, fewer tokens)")
print(f"CoT: {cot_r.sentiment} (with reasoning: '{cot_r.reasoning[:80]}...')")
print("\nFor simple classification, both give the same answer,")
print("but Predict uses fewer tokens.")

## Example 5: CoT in Custom Modules

Using ChainOfThought as part of a debate module that argues both sides.

In [ ]:
class DebateModule(dspy.Module):
    def __init__(self):
        super().__init__()

        class ArgumentFor(dspy.Signature):
            """Make the strongest argument in favor of the position."""
            position = dspy.InputField()
            argument = dspy.OutputField(desc="A compelling argument supporting the position")

        class ArgumentAgainst(dspy.Signature):
            """Make the strongest argument against the position."""
            position = dspy.InputField()
            argument = dspy.OutputField(desc="A compelling counter-argument")

        class Judge(dspy.Signature):
            """Evaluate both arguments and determine which is stronger."""
            position = dspy.InputField()
            for_argument = dspy.InputField()
            against_argument = dspy.InputField()
            verdict = dspy.OutputField(desc="Which side has the stronger argument and why")

        self.argue_for = dspy.ChainOfThought(ArgumentFor)
        self.argue_against = dspy.ChainOfThought(ArgumentAgainst)
        self.judge = dspy.ChainOfThought(Judge)

    def forward(self, position):
        pro = self.argue_for(position=position)
        con = self.argue_against(position=position)
        verdict = self.judge(
            position=position,
            for_argument=pro.argument,
            against_argument=con.argument
        )
        return dspy.Prediction(
            pro_argument=pro.argument,
            con_argument=con.argument,
            verdict=verdict.verdict
        )

debate = DebateModule()
result = debate(position="Schools should replace textbooks with tablets")

print_result(
    f"Position: Schools should replace textbooks with tablets\n\n"
    f"FOR: {result.pro_argument[:200]}...\n\n"
    f"AGAINST: {result.con_argument[:200]}...\n\n"
    f"VERDICT: {result.verdict[:200]}...",
    "Debate"
)

## Summary

**Key Takeaways:**
1. `ChainOfThought` adds a `reasoning` field to any signature
2. The model reasons step-by-step before giving the final answer
3. CoT significantly helps with complex, multi-step problems
4. For simple tasks, `Predict` may be sufficient and more efficient
5. Combine CoT with custom modules for sophisticated reasoning pipelines